# Tariff example usage
This notebook demonstrates how to use the `Tariff` class to calculate energy costs based on a given tariff structure. We will create a sample tariff and then use it to compute the cost of energy consumption over a specified period.

### 1. Creating an index
Most tariffs are based on one or more indexes. For example, a common structure is to have a fixed cost component and a variable cost component that depends on the day-ahead electricity price. Let's create an index for the day-ahead price using the `EntsoeDayAheadIndex`:

In [5]:
from os import environ

from energy_cost.index import EntsoeDayAheadIndex, Index

Index.register("Belpex15min", EntsoeDayAheadIndex(country_code="BE", api_key=environ["ENTSOE_API_KEY"]))

### 2. Defining a tariff
The easiest way to define a tariff is to specify it in a YAML file. This allows for a clear separation of the tariff definition from the code that uses it. you can find some real world examples in `data/tariffs/`. 

The yaml structure is as follows:

```yaml
supplier: "Foo" # The energy supplier providing the tariff
product: "Bar" # The specific product or plan offered by the supplier
components: 
  single_rate: # The metering type, eg. single_rate, tou_peak, tou_offpeak, ...
  # The ordered list of the valid price component, valid between its start and the start of the next component. The last component is valid until the end of time.
  - start: 2025-01-01T00:00:00+01 # The start date of the component, in ISO 8601 format
    constant_cost: 1.625 # The fixed cost component
    variable_costs: # The variable cost component, which depends on one or more indexes
    - index: Belpex15min # The index used for the variable cost
      scalar: 0.108 # The scalar applied to the index to calculate the variable cost
```

In [6]:
from energy_cost.tariff import Tariff

tariff = Tariff.from_yaml("../data/tariffs/EBEM/Groen_Dynamic.yml")
tariff

Tariff(supplier='EBEM', product='Groen Dynamic', components={<ComponentType.SINGLE_RATE: 'single_rate'>: [PriceComponent(start=datetime.datetime(2025, 1, 1, 0, 0, tzinfo=datetime.timezone(datetime.timedelta(seconds=3600))), constant_cost=1.625, variable_costs=[IndexBasedCost(index='Belpex15min', scalar=0.108)])], <ComponentType.INJECTION: 'injection'>: [PriceComponent(start=datetime.datetime(2025, 1, 1, 0, 0, tzinfo=datetime.timezone(datetime.timedelta(seconds=3600))), constant_cost=-1.1, variable_costs=[IndexBasedCost(index='Belpex15min', scalar=0.0925)])]})

### 3. Calculating cost per energy consumption in a given period
Once we have defined our tariff, we can use it to calculate the cost of energy consumption over a specified period. This is done by calling the `get_cost` method of the `Tariff` class, which returns a DataFrame with the cost for each time step in the given period.

In [7]:
import datetime as dt

from energy_cost.price_component import ComponentType

start = dt.datetime.fromisoformat("2026-03-08 00:00:00+01:00")
end = dt.datetime.fromisoformat("2026-03-10 00:00:00+01:00")
resolution = dt.timedelta(minutes=15)
tariff.get_cost(ComponentType.INJECTION, start, end, resolution)

,timestamp,value
0,2026-03-08 00:00:00+01:00,11.546600
1,2026-03-08 00:15:00+01:00,10.796425
2,2026-03-08 00:30:00+01:00,10.407000
3,2026-03-08 00:45:00+01:00,9.979650
4,2026-03-08 01:00:00+01:00,10.769600
...,...,...
187,2026-03-09 22:45:00+01:00,11.021200
188,2026-03-09 23:00:00+01:00,11.881450
189,2026-03-09 23:15:00+01:00,11.401375
190,2026-03-09 23:30:00+01:00,10.880600
